In [0]:
%pip install polars plotly

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import polars as pl
import plotly.express as px
import plotly.graph_objects as go

In [0]:
df_green= pl.from_pandas(spark.table('lumiere_capital.dbt_lumiere_capital.fct_portfolio_performance').toPandas())

In [0]:
print(f'✅ Loaded {len(df_green)} investment records for analysis')
display(df_green.head(5))

✅ Loaded 174714 investment records for analysis


transcations_id,company_id,company_name,sector,country,esg_score,amount_invested_eur,current_estimated_value,moic
i32,i32,str,str,str,f32,"decimal[9,2]",f64,f64
14282,1,"""SolGrowth_0""","""Offshore Wind""","""Sweden""",94.220001,2201315.58,1.0370e7,4.711
24921,2,"""HeliSystem_1""","""Circular Economy""","""Spain""",85.099998,2820733.12,1.2002e7,4.255
15010,3,"""SolGreen_2""","""Green Hydrogen""","""Sweden""",95.669998,1143675.74,5.4708e6,4.7835
22635,4,"""BioDesigns_3""","""Circular Economy""","""Denmark""",88.400002,896220.47,3.9613e6,4.42
24297,5,"""HeliSystem_4""","""AgriTech""","""Spain""",98.010002,283328.78,1.3885e6,4.9005


In [0]:
sector_summary = (
    df_green
    .group_by('sector')
    .agg([
        pl.len().alias('num_investments'),
        pl.col('amount_invested_eur').sum().alias('total_invested_eur'),
        pl.col('current_estimated_value').sum().alias('portfolio_value_eur'),
        pl.col('moic').mean().alias('avg_moic'),
        pl.col('esg_score').mean().alias('avg_esg_score')
    ])
    .with_columns([
        (pl.col('portfolio_value_eur') - pl.col('total_invested_eur')).alias('unrealized_pnl')
    ])
    .sort('avg_moic',descending=True)
)

display(sector_summary.head())

sector,num_investments,total_invested_eur,portfolio_value_eur,avg_moic,avg_esg_score,unrealized_pnl
str,u32,"decimal[9,2]",f64,f64,f32,f64
null,149714,149923210764.16,0.0,null,null,-1.4992e11
"""AgriTech""",4842,4805635521.88,1.9818e10,4.125214,82.504272,1.5013e10
"""Offshore Wind""",5299,5444355270.73,2.2389e10,4.116458,82.329163,1.6945e10
"""Circular Economy""",4922,4878916957.60,2.0054e10,4.114565,82.291306,1.5175e10
"""Green Hydrogen""",4953,4939985367.13,2.0303e10,4.111055,82.221092,1.5363e10


In [0]:
high_performers = (
    df_green
    .filter(pl.col('moic') > 2.0)
    .select(['company_name','sector','country','moic','esg_score'])
    .sort('moic',descending=True)
    .head(10)
)
display(high_performers.head())

company_name,sector,country,moic,esg_score
str,str,str,f64,f32
"""TerraSystem_1712""","""Green Hydrogen""","""Spain""",4.95,99.0
"""TerraSystem_1712""","""Green Hydrogen""","""Spain""",4.95,99.0
"""TerraSystem_1712""","""Green Hydrogen""","""Spain""",4.95,99.0
"""TerraSystem_1712""","""Green Hydrogen""","""Spain""",4.95,99.0
"""NeoDesigns_2502""","""Green Hydrogen""","""Denmark""",4.9495,98.989998


**Plotly Visualizations**

In [0]:
df_sectors = sector_summary.to_pandas().reset_index(drop=True)
df_sectors = df_sectors[df_sectors['sector'].notnull()]

fig_tree = px.treemap(
    df_sectors,
    path=['sector'],
    values='total_invested_eur',
    color='avg_moic',
    color_continuous_scale='Viridis',
    title='<b>Capital Allocation & Performance by Sector</b>',
    labels={'total_invested_eur':'Capital Invested (€)','avg_moic':'Average MOIC'}
)
fig_tree.update_layout(template='plotly_white')
fig_tree.show()

In [0]:
fig_dual = go.Figure()

# Bar for Avg MOIC
fig_dual.add_trace(go.Bar(
    x=sector_summary['sector'],
    y=sector_summary['avg_moic'],
    name='Avg MOIC',
    marker_color='#10b981'
))

# Line for Avg ESG Score
fig_dual.add_trace(go.Scatter(
    x=sector_summary['sector'],
    y=sector_summary['avg_esg_score'],
    name='Avg ESG Score',
    yaxis='y2',
    mode='lines+markers',
    line=dict(color='#3b82f6',width=4)
))

fig_dual.update_layout(
    title='<b>Returns vs. ESG Integrity by Sector </b>',
    yaxis=dict(title='Return Multiple (MOIC)'),
    yaxis2=dict(title='ESG Score (0-100)',overlaying='y',side='right'),
    template='plotly_dark',
    legend=dict(x=1.1,y=1)
)

fig_dual.show()